In [ ]:
import os
from pathlib import Path

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # go to ~/crossdem/ by jumping up twice
OUT_DIR = os.path.join(BASE_DIR, "datasets")

os.makedirs(OUT_DIR, exist_ok=True)

#with open(os.path.join(OUT_DIR, "output.txt"), "w") as f:
#    f.write("hello world")
#
#print(f"Written to {OUT_DIR}/output.txt")

In [ ]:
import re
import subprocess
import requests

def get_audio_url(page_url: str) -> str:
    """Scrape a Radio Radicale 'scheda' page for the embedded rtsp:// audio link."""
    resp = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    match = re.search(r'rtsp://[^"]+\.mp3', resp.text)
    if not match:
        raise ValueError("No rtsp audio link found on the page")
    return match.group(0)

def download_audio(rtsp_url: str, output_file: str = "output.mp3"):
    """Use ffmpeg to pull the audio stream down to a local file."""
    cmd = [
        "ffmpeg",
        "-y",                # overwrite if exists
        "-i", rtsp_url,
        "-acodec", "copy",   # no re-encoding, just remux
        output_file,
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    page = "https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617"
    audio_url = get_audio_url(page)
    print("Found:", audio_url)
    download_audio(audio_url, f"{OUT_DIR}fanfani_1987.mp3")

In [2]:
import yt_dlp

url = "https://www.radioradicale.it/scheda/20785?i=2726016"

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": "%(title)s.%(ext)s",
    "postprocessors": [{
        "key": "FFmpegExtractAudio",
        "preferredcodec": "mp3",   # change to "best" to keep original format without re-encoding
        "preferredquality": "192",
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/20785?i=2726016
[RadioRadicale] 20785: Downloading webpage
[RadioRadicale] 20785-0: Downloading m3u8 information
[info] 20785: Downloading 1 format(s): 63
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 592
[download] Destination: Crisi di governo.m4a
[download] 100% of   54.19MiB in 00:01:07 at 819.25KiB/s                 
[FixupM3u8] Fixing MPEG-TS in MP4 container of "Crisi di governo.m4a"
[ExtractAudio] Destination: Crisi di governo.mp3
Deleting original file Crisi di governo.m4a (pass -k to keep)
